In [1]:
# import sys
# from pathlib import Path

# # go one level up from notebooks/ to project root
# project_root = Path().resolve().parent

# sys.path.append(str(project_root))

In [2]:
import os
os.chdir("..")  # move from notebooks/ to project root

In [3]:
import pandas as pd
import numpy as np

In [4]:
from src.models.dataset import EvaluationDataset
from src.prompts.prompt_optimizer import optimize_prompt
from src.prompts.utils import evaluate_prompt

In [5]:
dataset = EvaluationDataset.from_csv(path="data/datasets/train_df.csv")

models = ["mistral", "gemma2"]

In [6]:
dataset.samples = dataset.samples#[:50]

In [7]:
len(dataset.samples)

180

In [8]:
initial_prompt = """
You are a helpful assistant.

Answer the question accurately and clearly.

Question:
{query}

Answer:
"""

In [9]:
results = optimize_prompt("mistral", dataset, initial_prompt)

[CACHE MISS] mistral
{"timestamp": "2026-05-28T16:07:14.461676", "level": "INFO", "message": "Starting inference with model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-28T16:07:33.713257", "level": "INFO", "message": "Inference completed model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-28T16:07:33.716291", "level": "INFO", "message": "Starting inference with model=llama3", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-28T16:07:44.483393", "level": "INFO", "message": "Inference completed model=llama3", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-28T16:07:44.484986", "level": "INFO", "message": "Starting inference with model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-28T16:07:48.606893", "level": "INFO", "message": "Inference completed model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"t

In [10]:
results

{'best_prompt': 'You are a knowledgeable assistant.\n\nAccurately answer the question based on factual information, providing clear and concise responses.\n\nQuestion:\n{query}\n\nAnswer:\n\n\nPlease note that I have only returned the rewritten prompt and not provided any explanations.',
 'best_score': np.float64(0.8288888888888891),
 'history': [{'iteration': 0,
   'score': np.float64(0.8263888888888891),
   'prompt': '\nYou are a helpful assistant.\n\nAnswer the question accurately and clearly.\n\nQuestion:\n{query}\n\nAnswer:\n'},
  {'iteration': 1,
   'score': np.float64(0.815388888888889),
   'prompt': 'Here is the rewritten prompt:\n\nYou are a helpful assistant.\n\nAccurately and clearly answer the question about {query}, considering both factual correctness and reasoning quality, with minimal hallucinations and concise instructions.\n\nNote: The original prompt\'s vague instruction ("Answer the question accurately and clearly.") has been replaced with a more specific and goal-o

## Analysis - Summary

* Due to time constraints, the above prompt optimizer was run only for 5 iterations based on a subset (50 samples) of the train dataset.
* Given more time, we would run the optimizer for more iterations and with more samples.
* I would also experiment with the prompt given to the better model (llama3)


**Analysis of Prompt Modifications**

* The strongest-performing prompts introduced clearer task framing while remaining concise and structurally stable.
* Unlike earlier optimization attempts that introduced excessive constraints and verbose factuality instructions, successful prompts preserved simplicity while improving instruction specificity.

The experiments suggest that weaker models benefit from:
* Lightweight structure
* Explicit response expectations
* Concise reasoning guidance

rather than heavily engineered prompts.

## Analysis - Conclusions

**Optimized Prompt Findings**

The prompt optimization experiments demonstrated that prompt quality is highly sensitive to instruction structure and wording.
The strongest-performing prompt was the original concise baseline prompt:

**"""**

**You are a helpful assistant.**

**Answer the question accurately and clearly by providing a concise definition or explanation.**

**Question:**

**{query}**

**Explain:**

**"""**

**Key Finding**

Prompt optimization was most effective when:
* Improving instruction clarity
* Reducing ambiguity
* Preserving concise structure

Overly restrictive prompts degraded performance, while lightweight structural guidance improved answer quality and consistency.

## Next steps / Improvements

Potential Next Steps and Improvements

1. Constrained Prompt Optimization

The current optimizer allowed unrestricted prompt rewriting, which caused prompt drift and optimization collapse. A more stable approach would constrain optimization to specific prompt components while preserving a fixed template structure.

**You are a {ROLE}.**

**{REASONING_STRATEGY}**

**Question:**

**{query}**

**Answer:**


Only selected components such as role framing, reasoning instructions, and formatting constraints would be mutated during optimization.

2. Structured Prompt Search Space

Instead of free-form rewriting, define a controlled search space:
- Role definitions:
  - helpful assistant
  - expert analyst
  - precise reasoner
- Reasoning strategies:
  - answer clearly
  - think step by step
  - explain briefly before final answer
- Output formatting:
  - concise answer
  - bullet points
  - final answer only

This would improve optimization stability and reproducibility.

3. Multi-Objective Optimization

The current optimization objective focused only on correctness. Future versions could optimize:
- correctness
- latency
- token usage
- inference cost

Example utility function:
utility = correctness - λ * latency - γ * token_cost

4. Better Router Utility Formulation

The correctness / latency ratio strongly favored low-latency models. A weighted utility function would likely produce better routing behavior:
utility = α * correctness - β * latency

This would prevent latency from dominating the routing objective.

5. Difficulty-Aware Routing

Add a query difficulty estimator to classify queries as:
- easy
- medium
- hard

Harder queries could be routed to stronger models, while easier queries could use cheaper and faster models.

6. Ensemble-Based Fallbacks

When router confidence is low, multiple models could be queried and aggregated using:
- majority voting
- LLM-as-a-judge ranking
- confidence-weighted selection

This would improve robustness for ambiguous or difficult queries.

7. Online Learning and Continuous Adaptation

The router could continuously improve using production feedback logs:
(query, selected_model, correctness, latency)

This would allow periodic retraining and adaptation to distribution shifts in user queries.

8. Prompt Transferability Studies

Future work could evaluate whether prompts optimized for one model transfer effectively to other models. This would help distinguish:
- model-specific prompt engineering
vs.
- universally beneficial prompt structures

9. Contextual Bandit Routing

A more advanced routing formulation would treat model selection as a contextual bandit problem:
- state = query embedding
- action = selected model
- reward = correctness - cost

This would enable adaptive exploration/exploitation strategies during deployment.

10. Improved Prompt Optimization Algorithms

Future optimization methods could include:
- genetic algorithms
- beam search over prompts
- Bayesian optimization
- reinforcement learning based prompt search

These approaches may explore the prompt space more effectively than greedy iterative rewriting.

## Evaluate performance on test dataset

In [11]:
test_dataset = EvaluationDataset.from_csv(path="data/datasets/test_df.csv")

In [12]:
prompt_template = """
You are a helpful assistant.

Answer the question accurately and clearly.

Question:
{query}

Answer:
"""

score, failures = evaluate_prompt(
    "mistral", prompt_template, test_dataset,
)

score

[CACHE HIT] mistral
[✓] Fully loaded from cache


np.float64(0.8057377049180326)

In [13]:
prompt_template = """
{query}
"""

score, failures = evaluate_prompt(
    "mistral", prompt_template, test_dataset,
)

score

[CACHE HIT] mistral
[✓] Fully loaded from cache


np.float64(0.821311475409836)

In [17]:
def clean_prompt(prompt: str):

    lines = prompt.split("\n")

    filtered = []

    for line in lines:

        if line.startswith("Here is"):
            continue

        if line.startswith("Note:"):
            continue

        if line.startswith("Please note "):
            continue

        filtered.append(line)

    return "\n".join(filtered).strip() + "\n"

In [20]:
prompt_template = clean_prompt(results["best_prompt"])

score, failures = evaluate_prompt(
    "mistral", prompt_template, test_dataset,
)

score

[CACHE MISS] mistral
{"timestamp": "2026-05-29T04:33:21.424131", "level": "INFO", "message": "Starting inference with model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-29T04:33:41.925086", "level": "INFO", "message": "Inference completed model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-29T04:33:41.928514", "level": "INFO", "message": "Starting inference with model=llama3", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-29T04:33:51.508312", "level": "INFO", "message": "Inference completed model=llama3", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-29T04:33:51.509802", "level": "INFO", "message": "Starting inference with model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"timestamp": "2026-05-29T04:33:59.082321", "level": "INFO", "message": "Inference completed model=mistral", "module": "ollama_client", "request_id": "unknown"}
{"t

np.float64(0.8163934426229508)

In [21]:
prompt_template

'You are a knowledgeable assistant.\n\nAccurately answer the question based on factual information, providing clear and concise responses.\n\nQuestion:\n{query}\n\nAnswer:\n'

## Conclusions on Test Dataset

* The optimized prompt based on a subset of the train dataset outperformed both the initial prompt as well as the plain prompt including only the query. More specifically:
* 1. Optimized Prompt Score: 0.8270
  2. Simple Prompt Score: 0.8213
  3. Initial Prompt Score: 0.8057